# DuckDB on the real Alaska fuel network

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jccheesman/alaska-fuel-agents-demo/blob/main/notebooks/00_duckdb_fuel_network.ipynb)

The agents in the harness never see a map. They see **one DuckDB file**, `fuel_network.duckdb`
(29.9 MB), through a handful of tools. This notebook opens that same file directly and shows what
DuckDB brings to the problem: plain SQL over a million weight rows, table macros and views that
encode the season, and community extensions that put **routing (duckrouting)**, **graph analytics
(Onager)**, **H3 tiles**, **spatial types** and **full-text search** inside the database.

Boxes marked **Callout** say where the harness itself relies on the feature.

Runs on a CPU runtime in a few minutes. Needs a read-only GitHub token in Colab Secrets (see the first cell);
the harness repository is private while the study is under way.

## 1 · Get the database

In [ ]:
# The harness repository is private while the study runs, so this notebook
# reads a GitHub token from Colab's Secrets panel (the key icon on the left):
#   name: GH_TOKEN   value: a fine-grained personal access token with
#   read-only "Contents" permission on jccheesman/agentic_harness_for_alaska_fuel_network
# Nothing is written back to GitHub. Outside Colab, export GH_TOKEN instead.
import os
try:
    from google.colab import userdata  # type: ignore
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    GH_TOKEN = os.environ.get("GH_TOKEN", "")
assert GH_TOKEN, "Add GH_TOKEN to Colab Secrets (or export it) and re-run this cell."

HARNESS_COMMIT = "8eef6ac13d36ac1f23d742466ec0789ca3fe9c8c"   # the commit this demo is pinned to
DB_SHA256 = "a8b38b71dc7d16b9c3eecf0d83714552edbb9aaa1f81c4055e0fb5fd8a13d3d4"
REPO = "jccheesman/agentic_harness_for_alaska_fuel_network"
print("token present; pinned commit", HARNESS_COMMIT[:12])

In [ ]:
# DuckDB 1.5.6 is the version the database was written with (schema_meta.duckdb_min)
# and the one the extensions were validated on by the harness's VM tests.
%pip -q install duckdb==1.5.6 pandas matplotlib requests
import duckdb, pandas as pd
print("duckdb", duckdb.__version__)

In [ ]:
# Download just the database (29.9 MB) through the GitHub contents API at the
# pinned commit, then verify its hash against the one the demo was built from.
import hashlib, pathlib, requests

db = pathlib.Path("fuel_network.duckdb")
if not db.exists():
    url = f"https://api.github.com/repos/{REPO}/contents/fuel_network.duckdb?ref={HARNESS_COMMIT}"
    r = requests.get(url, headers={"Authorization": f"Bearer {GH_TOKEN}",
                                   "Accept": "application/vnd.github.raw+json"}, stream=True, timeout=120)
    r.raise_for_status()
    with open(db, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
sha = hashlib.sha256(db.read_bytes()).hexdigest()
print(db, f"{db.stat().st_size/1e6:.1f} MB", "sha256", sha[:16])
assert sha == DB_SHA256, "database hash differs from the pinned one"

## 2 · DuckDB in one minute

> **Callout — why DuckDB here.** One file, no server, columnar storage, SQL that runs in-process
> next to Python. The whole network (84,094 nodes, 93,049 edges, 1.1 million edge-month weights and
> as many costs) opens read-only in milliseconds, so the harness can hand every agent process its own
> connection and never worry about state leaking between runs.

In [ ]:
con = duckdb.connect("fuel_network.duckdb", read_only=True)
con.sql("SHOW TABLES")

In [ ]:
# Every table with its row count, in one query over the catalog
con.sql('''
SELECT table_name, estimated_size AS rows, column_count
FROM duckdb_tables() ORDER BY rows DESC
''')

In [ ]:
con.sql("SELECT * FROM schema_meta")   # what the harness stamped into the file

In [ ]:
con.sql("DESCRIBE network_edges")

In [ ]:
# SUMMARIZE profiles a whole table (min/max/approx distinct/nulls) in one call
con.sql("SUMMARIZE SELECT latitude, longitude, delivery_method_1, qc_status FROM facilities")

## 3 · The graph as tables

Nodes and edges are ordinary tables. **Edges have no geometry of their own**: a line is drawn from
one node's coordinates to the other's. Every edge gets one row per month in `edge_month_weights`
(friction and passability) and one in `edge_costs` (modelled cost per gallon).

> **Callout — season is data, not code.** "Ice roads are open January to March" is not an `if`
> statement anywhere in the harness. It is the `passable` flag on 1,282 ice-road edges in three of
> their twelve monthly rows. That is why a disruption can be expressed the same way: as a filter.

In [ ]:
# Passable edges per mode per month. PIVOT turns months into columns.
con.sql('''
PIVOT (SELECT mode, month, count(*) AS n FROM edge_month_weights WHERE passable GROUP BY 1, 2)
ON month USING sum(n) GROUP BY mode ORDER BY mode
''')

In [ ]:
# The modes table: edge types -> report mode, and the cost rates the costs were built with
con.sql("SELECT mode, report_mode, edge_types, rate_per_gallon_mile, transfer_fee_per_gallon FROM modes")

In [ ]:
# A hub is a node with is_hub = true. Here is Nome, resolved by community name.
nome = con.sql("SELECT hub_id, hub_pid, node_id, community, region, hub_type, deliv_meth FROM hub_community WHERE community = 'Nome'").df()
anch = con.sql("SELECT hub_id, hub_pid, node_id, community, region, hub_type FROM hub_community WHERE hub_pid = 'h000698'").df()
display(nome, anch)
NOME_NODE, ANCH_NODE = int(nome.node_id[0]), int(anch.node_id[0])

## 4 · Macros and views the harness built

`schema_stage0.py` added a table macro and three views on top of the four base tables. The macro
`passable_edges(month, modes)` is the one every router uses.

> **Callout — "a disruption is a WHERE clause, never a mutation."** The harness module that
> created this macro says exactly that. To close the Port of Nome you do not delete anything: you
> run the same query with `AND edge_id NOT IN (...)`. The data stays frozen and auditable.

In [ ]:
# the macro, as DuckDB stores it
print(con.sql("SELECT macro_definition FROM duckdb_functions() WHERE function_name = 'passable_edges'").fetchone()[0])

In [ ]:
ALL_MODES = ['Road', 'Barge', 'Plane', 'IceRoad', 'Transfer']
con.sql("SELECT mode, count(*) AS edges, round(avg(cost_per_gallon), 4) AS avg_cost FROM passable_edges(9, ?) GROUP BY 1 ORDER BY 1", params=[ALL_MODES])

In [ ]:
# hub_access: which modes reach each hub's own node, and in which months
con.sql("SELECT hub_pid, community, mode, n_edges, months_passable FROM hub_access WHERE community IN ('Nome', 'Unalakleet', 'Shishmaref') ORDER BY 2, 3")

In [ ]:
# corridor_graph: the coarse graph (hub to hub, one mode per corridor) that the analytics tools use
con.sql("SELECT month, count(*) AS corridors, sum(passable::int) AS passable FROM corridor_graph GROUP BY 1 ORDER BY 1")

## 5 · Routing inside the database with **duckrouting**

`duckrouting` is a community extension that exposes pgRouting-style functions over any SQL query
that returns `id, source, target, cost, reverse_cost`. The harness's `graph.py` has a
`DuckroutingEngine` that builds exactly the query below.

> **Callout — the accepted route engine.** On the study VM, `DOE_MAS_GRAPH_ENGINE=duckrouting`
> read the reference routes identical to the pure-Python oracle, edge for edge, in two runs in a
> row, and passed the 27 router tests (INTERNAL_NOTES, 2026-10-14). It was accepted. The Python
> oracle stays the default because the two are byte-equal and the oracle needs no extension.

In [ ]:
def load_ext(name, community=True):
    try:
        con.execute(f"LOAD {name}")
        return True
    except Exception:
        try:
            con.execute(f"INSTALL {name}{' FROM community' if community else ''}")
            con.execute(f"LOAD {name}")
            return True
        except Exception as e:
            print(f"[{name}] could not be installed here: {str(e)[:160]}")
            return False

HAVE_ROUTING = load_ext("duckrouting")
print("duckrouting loaded:", HAVE_ROUTING)

In [ ]:
# the edge SQL duckrouting wants, built the way graph.DuckroutingEngine._edges_sql builds it
def edges_sql(month=9, weight="cost_per_gallon", avoid=()):
    modes = ", ".join(f"'{m}'" for m in ALL_MODES)
    sql = (f"SELECT edge_id AS id, from_node AS source, to_node AS target, "
           f"{weight} AS cost, {weight} AS reverse_cost "
           f"FROM passable_edges({month}, [{modes}]) WHERE {weight} IS NOT NULL")
    if avoid:
        sql += " AND edge_id NOT IN (" + ", ".join(str(int(e)) for e in avoid) + ")"
    return sql

def dijkstra(src, dst, month=9, avoid=()):
    rows = con.execute("SELECT node, edge, agg_cost FROM duckrouting_dijkstra(?, ?, ?)",
                       [edges_sql(month, avoid=avoid), int(src), int(dst)]).fetchall()
    edges = [int(r[1]) for r in rows if int(r[1]) != -1]
    return edges, (rows[-1][2] if rows else None)

if HAVE_ROUTING:
    base_edges, base_cost = dijkstra(ANCH_NODE, NOME_NODE)
    print(f"Anchorage -> Nome, September, cheapest: {len(base_edges)} edges, ${base_cost:.3f}/gal")

In [ ]:
# Reference answer in the harness: 4,899 edges and $1.848/gal (tests/reference_answers/routes.json).
# The route's modes, in travel order, and its length:
if HAVE_ROUTING:
    con.register("route_edges", pd.DataFrame({"seq": range(len(base_edges)), "edge_id": base_edges}))
    display(con.sql('''
        SELECT m.report_mode AS mode, count(*) AS edges, round(sum(e.length_m)/1000, 1) AS km
        FROM route_edges r JOIN network_edges e USING (edge_id)
        JOIN modes m ON list_contains(m.edge_types, e.type)
        GROUP BY 1 ORDER BY min(r.seq)
    '''))

### The storm closes the port

Disruption **D2** voids three transfer edges at Nome (`edge:e091382`, `edge:e094423`,
`edge:e097314`), the Nome hub, nine facilities and three corridors. Removing the three edges from
the month's graph is the whole reroute.

In [ ]:
D2_EDGE_PIDS = ['e091382', 'e094423', 'e097314']
d2_ids = [r[0] for r in con.execute("SELECT edge_id FROM network_edges WHERE edge_pid IN (SELECT unnest(?::VARCHAR[]))", [D2_EDGE_PIDS]).fetchall()]
print("internal edge ids voided by D2:", d2_ids)
if HAVE_ROUTING:
    post_edges, post_cost = dijkstra(ANCH_NODE, NOME_NODE, avoid=d2_ids)
    print(f"after the closure: {len(post_edges)} edges, ${post_cost:.3f}/gal  (reference: 4,922 edges, $2.066/gal)")
    print("baseline used a voided edge:", bool(set(base_edges) & set(d2_ids)), "| reroute uses one:", bool(set(post_edges) & set(d2_ids)))

In [ ]:
# k shortest paths (Yen) and a cost-budget reach, both straight from SQL.
# duckrouting follows pgRouting's KSP column layout: seq, path_id, path_seq, node, edge, cost, agg_cost.
if HAVE_ROUTING:
    ksp = con.execute("SELECT * FROM duckrouting_ksp(?, ?, ?, ?)", [edges_sql(9), ANCH_NODE, NOME_NODE, 3]).df()
    display(ksp.groupby(ksp.columns[1]).agg(edges=(ksp.columns[4], 'count'), cost=(ksp.columns[-1], 'max')))
    reach = con.execute("SELECT count(*) AS nodes_within_budget FROM duckrouting_driving_distance(?, ?, ?)", [edges_sql(9), NOME_NODE, 0.5]).df()
    display(reach)   # nodes reachable from Nome for at most $0.50/gal in September

In [ ]:
# Connected components per month: how fragmented is the passable network in April vs September?
if HAVE_ROUTING:
    for month in (4, 9):
        comps = con.execute("SELECT * FROM duckrouting_connected_components(?)", [edges_sql(month)]).df()
        col = [c for c in comps.columns if 'comp' in c.lower()][0]
        sizes = comps.groupby(col).size().sort_values(ascending=False)
        print(f"month {month}: {len(sizes)} components, largest {sizes.iloc[0]} nodes")

> **Callout — how an extension earns trust.** The VM tests record where duckrouting disagrees
> with the oracle and why: its `duckrouting_bridges` (Boost biconnected components) misreads
> parallel corridors and self-loops, so bridges are compared on a simplified graph; its betweenness
> returned NaN on this graph, so the oracle's `betweenness` stays the engine. Extensions are
> adopted function by function, against a reference, never wholesale.

## 6 · Graph analytics inside SQL with **Onager**

The `region` and `global_topology` tools work on the ~2,300-corridor September graph, not on the
93,000 edges. Onager's table functions take a subquery of `(source, target)` rows.

> **Callout — Onager in the harness.** `onager_cmm_louvain` and `onager_sub_k_hop` are checked
> against the pure-Python `graph_analytics.py` on the VM. Onager's Louvain takes no weight column,
> so the harness compares it with the oracle's *unweighted* Louvain (modularity 0.862 vs 0.861) and
> keeps the inverse-cost community the `region` tool reports on the Python side. The owner also
> ruled that a graph-theory "bridge" is called a **failure corridor** everywhere agents can see it,
> because in this network a bridge is a road structure over water.

In [ ]:
HAVE_ONAGER = load_ext("onager")
MONTH = 9
ONAGER_EDGES = f"(SELECT node_a AS source, node_b AS target FROM corridor_graph WHERE month = {MONTH} AND passable AND cost_per_gallon IS NOT NULL)"
if HAVE_ONAGER:
    comps = con.sql(f"SELECT * FROM onager_cmm_components({ONAGER_EDGES})").df()
    print("components:", comps.iloc[:, -1].nunique(), "| nodes:", len(comps))
    louv = con.sql(f"SELECT * FROM onager_cmm_louvain({ONAGER_EDGES}, seed := 1)").df()
    print("Louvain communities:", louv.iloc[:, -1].nunique())
    khop = con.sql(f"SELECT * FROM onager_sub_k_hop({ONAGER_EDGES}, start := {NOME_NODE}, k := 2)").df()
    print("nodes within 2 corridors of Nome:", len(khop))
    display(khop.head())

In [ ]:
# Which Louvain community does Nome fall in, and which hubs share it?
if HAVE_ONAGER:
    node_col, comm_col = louv.columns[0], louv.columns[-1]
    con.register("louv", louv)
    nome_comm = con.execute(f"SELECT {comm_col} FROM louv WHERE {node_col} = ?", [NOME_NODE]).fetchone()
    if nome_comm:
        display(con.sql(f'''
            SELECT h.community, h.hub_pid, h.region FROM louv l JOIN hub_community h ON h.node_id = l.{node_col}
            WHERE l.{comm_col} = {nome_comm[0]} ORDER BY 1
        '''))

## 7 · **H3** tiles: the honest spatial key

Every node, hub and facility carries an H3 cell at resolution 5 (about 17 km across) and
resolution 4. Agents cite cells in receipts and the L3 context index is grounded to them.

> **Callout — an index, never a filter.** `schema_tiles.py` writes the `tiles` table with the H3
> DuckDB extension where it loads and the `h3` Python package where it cannot, and records which in
> `schema_meta.tiles_source`. Tiles index pages and lookups; the router never filters on them.

In [ ]:
HAVE_H3 = load_ext("h3")
if HAVE_H3:
    mism = con.sql('''
        SELECT count(*) FROM tiles
        WHERE h3_latlng_to_cell(lat, lon, 5) <> h3_r5 OR h3_cell_to_parent(h3_r5, 4) <> h3_r4
    ''').fetchone()[0]
    print("stored tiles that disagree with the extension:", mism, "(the VM test expects 0)")
    nome_cell = con.sql("SELECT h3_r5_str FROM tiles WHERE kind='node' AND ref_id = ?", params=[NOME_NODE]).fetchone()[0]
    print("Nome's r5 cell:", nome_cell)
    display(con.sql(f'''
        WITH ring AS (SELECT unnest(h3_grid_disk(h3_string_to_h3('{nome_cell}'), 1)) AS cell)
        SELECT t.kind, count(*) AS n FROM tiles t JOIN ring r ON t.h3_r5 = r.cell GROUP BY 1
    '''))   # what sits within one ring of Nome's cell

In [ ]:
# facilities per r5 cell: the densest cells in the state
con.sql('''
SELECT h3_r5_str AS cell, count(*) AS facilities, any_value(f.community_name) AS a_community
FROM tiles t JOIN facilities f ON t.kind = 'facility' AND t.ref_id = f.facility_id
GROUP BY 1 ORDER BY 2 DESC LIMIT 8
''')

## 8 · The **spatial** extension: coordinates, reprojection, GeoJSON out

Node coordinates are stored in EPSG:3338 (NAD83 / Alaska Albers, metres). The spatial extension
reprojects in SQL and exports GeoJSON that any GIS or web map can read.

In [ ]:
HAVE_SPATIAL = load_ext("spatial", community=False)
if HAVE_SPATIAL:
    display(con.sql('''
        SELECT n.node_id,
               ST_X(ST_Transform(ST_Point(n.x, n.y), 'EPSG:3338', 'EPSG:4326', always_xy := true)) AS lon_sql,
               t.lon AS lon_stored,
               ST_Y(ST_Transform(ST_Point(n.x, n.y), 'EPSG:3338', 'EPSG:4326', always_xy := true)) AS lat_sql,
               t.lat AS lat_stored
        FROM network_nodes n JOIN tiles t ON t.kind='node' AND t.ref_id=n.node_id
        WHERE n.is_hub LIMIT 5
    '''))

In [ ]:
# export the hubs as GeoJSON (GDAL driver) for QGIS or a web map
if HAVE_SPATIAL:
    con.sql('''
        COPY (SELECT hub_pid, community, region, hub_type,
                     ST_Transform(ST_Point(x, y), 'EPSG:3338', 'EPSG:4326', always_xy := true) AS geom
              FROM hub_community)
        TO 'hubs.geojson' WITH (FORMAT GDAL, DRIVER 'GeoJSON')
    ''')
    import os; print("hubs.geojson", os.path.getsize("hubs.geojson"), "bytes")

In [ ]:
# a quick map: baseline vs post-closure route, voided edges in red (palette from report_figures.py)
import matplotlib.pyplot as plt
PAL = {"road": "#2a78d6", "barge": "#eb6834", "plane": "#1baf7a", "ice_road": "#4a3aa7", "transfer": "#7a6a58"}
node_ll = dict(con.sql("SELECT ref_id, [lon, lat] FROM tiles WHERE kind='node'").fetchall())
ends = dict(con.sql("SELECT edge_id, [from_node, to_node] FROM network_edges").fetchall())

def draw(ax, edges, **kw):
    for e in edges:
        a, b = ends[e]
        (x1, y1), (x2, y2) = node_ll[a], node_ll[b]
        ax.plot([x1, x2], [y1, y2], **kw)

if HAVE_ROUTING:
    fig, ax = plt.subplots(figsize=(9, 6))
    hubs = con.sql("SELECT x, y, community FROM hub_community").df()
    ax.scatter(*zip(*[node_ll[n] for n in con.sql("SELECT node_id FROM hub_community").df().node_id]), s=4, c="#9aa", zorder=1)
    draw(ax, base_edges, color="#9aa", lw=2.5, alpha=0.6, label="before")
    draw(ax, post_edges, color=PAL["barge"], lw=1.4, label="after D2")
    draw(ax, d2_ids, color="#d03b3b", lw=5, label="voided edges")
    for n, name in ((ANCH_NODE, "Anchorage"), (NOME_NODE, "Nome")):
        ax.annotate(name, node_ll[n], fontsize=9, fontweight="bold")
    ax.set_xlim(-171, -147); ax.set_ylim(59.5, 66.5); ax.set_title("Anchorage → Nome, September: before and after the Port of Nome closure")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    h, l = ax.get_legend_handles_labels(); ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), loc="lower left")
    plt.show()

## 9 · DuckPGQ: the graph as a property graph (optional)

DuckPGQ adds SQL/PGQ `MATCH` patterns over ordinary tables. The harness only checks that it loads
(`tools/vm_first_tests`, test 6); it is shown here because it is the most readable way to ask a
graph question in SQL. Skip-safe.

In [ ]:
HAVE_PGQ = load_ext("duckpgq")
if HAVE_PGQ:
    try:
        pg = duckdb.connect()   # a scratch in-memory db we may write to
        pg.execute("LOAD duckpgq")
        pg.execute("ATTACH 'fuel_network.duckdb' AS fn (READ_ONLY)")
        pg.execute("CREATE TABLE hubs AS SELECT node_id, hub_pid, community FROM fn.hub_community")
        pg.execute(f'''CREATE TABLE cors AS SELECT node_a, node_b, corridor_pid, mode, cost_per_gallon
                       FROM fn.corridor_graph WHERE month = {MONTH} AND passable AND node_a IN (SELECT node_id FROM hubs) AND node_b IN (SELECT node_id FROM hubs)''')
        pg.execute('''CREATE PROPERTY GRAPH fuel
                      VERTEX TABLES (hubs PROPERTIES (node_id, hub_pid, community) LABEL hub)
                      EDGE TABLES (cors SOURCE KEY (node_a) REFERENCES hubs (node_id)
                                        DESTINATION KEY (node_b) REFERENCES hubs (node_id)
                                        PROPERTIES (corridor_pid, mode, cost_per_gallon) LABEL corridor)''')
        display(pg.sql('''FROM GRAPH_TABLE (fuel
                          MATCH (a:hub WHERE a.community = 'Nome')-[c:corridor]-(b:hub)
                          COLUMNS (b.community, b.hub_pid, c.mode, c.cost_per_gallon)) ORDER BY 4 LIMIT 10'''))
    except Exception as e:
        print("DuckPGQ loaded but the pattern query failed here:", str(e)[:200])

## 10 · Full-text search over the concept map pages

> **Callout — the L3 `search_context` tool.** Context documents are cut into chunks and ranked with
> BM25, in Python by default or with DuckDB's `fts` extension when it loads; every result records
> which engine answered. Here the same mechanism runs over a few dozen hub pages of the concept map.

In [ ]:
# fetch ~40 hub pages of the OKF concept map through the same API (small markdown files)
import json
def gh_json(path):
    r = requests.get(f"https://api.github.com/repos/{REPO}/contents/{path}?ref={HARNESS_COMMIT}",
                     headers={"Authorization": f"Bearer {GH_TOKEN}", "Accept": "application/vnd.github+json"}, timeout=60)
    r.raise_for_status(); return r.json()
def gh_raw(path):
    r = requests.get(f"https://api.github.com/repos/{REPO}/contents/{path}?ref={HARNESS_COMMIT}",
                     headers={"Authorization": f"Bearer {GH_TOKEN}", "Accept": "application/vnd.github.raw+json"}, timeout=60)
    r.raise_for_status(); return r.text
index = json.loads(gh_raw("okf/_index.json"))
hub_pages = [v["path"] for k, v in index.items() if v["kind"] == "hub"][:40]
pages = pd.DataFrame({"path": hub_pages, "body": [gh_raw("okf/" + p) for p in hub_pages]})
print(len(pages), "pages; e.g.", pages.path[0])

In [ ]:
HAVE_FTS = load_ext("fts", community=False)
if HAVE_FTS:
    mem = duckdb.connect()
    mem.execute("LOAD fts")
    mem.register("pages_src", pages)
    mem.execute("CREATE TABLE pages AS SELECT row_number() OVER () AS i, path, body FROM pages_src")
    mem.execute("PRAGMA create_fts_index('pages', 'i', 'body', stemmer='none', stopwords='none', ignore='(\\.|[^a-z0-9])+', lower=true)")
    display(mem.sql('''
        SELECT path, round(fts_main_pages.match_bm25(i, 'barge transfer receiver'), 3) AS score
        FROM pages WHERE score IS NOT NULL ORDER BY score DESC LIMIT 5
    '''))

## 11 · Where the agents meet the database

The tools an agent calls are thin wrappers around queries like the ones above, plus one thing the
raw database does not give you: a **receipt**. Every call is logged with an `r-####` id, its
arguments and a hash of its result, and a report may only cite receipts the run actually received.
The second notebook calls those tools directly.

A last export, for anyone who wants the hub table in another tool:

In [ ]:
con.sql("COPY (SELECT * FROM hub_community) TO 'hub_community.parquet' (FORMAT PARQUET)")
import os; print("hub_community.parquet", os.path.getsize("hub_community.parquet"), "bytes")
con.close()